# Credit Default Prediction & Risk Explainability

这个 Notebook 是给展示和复盘用的版本。它要讲清楚三件事：

1. **项目研究什么**：预测借款人未来两年是否会严重逾期。
2. **模型怎么做**：清洗数据、做风险特征、训练并比较多个模型。
3. **业务怎么用**：根据风险分数设置审批阈值，平衡通过率和坏账风险。

一句话总结：

> 这是一个信贷风控项目，用客户的收入、负债、信用使用率和历史逾期信息，预测未来两年违约风险，并把模型结果转化成审批策略。

## 1. 项目背景

银行或 FinTech 平台在发放贷款前，通常需要回答：

> 这个客户未来会不会违约？

如果客户风险高，业务上可以采取：

- 拒绝贷款申请
- 降低额度
- 转人工审核
- 提高贷后监控

本项目使用 Kaggle **Give Me Some Credit** 数据集。每一行代表一个借款人，目标变量是：

```text
SeriousDlqin2yrs
```

含义：

- `0`：未来两年内没有严重逾期
- `1`：未来两年内发生严重逾期

In [ ]:
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw" / "cs-training.csv").exists():
    ROOT = ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = ROOT / "data" / "raw" / "cs-training.csv"
REPORT_DIR = ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
MODEL_DIR = REPORT_DIR / "models"
TARGET = "SeriousDlqin2yrs"
RANDOM_STATE = 42

## 2. 读取数据

先看数据规模、字段，以及目标变量违约率。

违约率很重要，因为信贷违约通常是**样本不平衡问题**：大多数客户不会违约，少数客户会违约。

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw.drop(columns=["Unnamed: 0"], errors="ignore")

print("Shape:", df_raw.shape)
display(df_raw.head())

In [ ]:
default_rate = df_raw[TARGET].mean()
print(f"Default rate: {default_rate:.4%}")
print(df_raw[TARGET].value_counts(normalize=True).rename("share"))

## 3. 字段理解

核心字段可以分成几类：

| 类型 | 字段例子 | 风控含义 |
|---|---|---|
| 收入能力 | `MonthlyIncome` | 收入越稳定，偿债能力通常越强 |
| 负债压力 | `DebtRatio` | 负债越高，还款压力越大 |
| 信用使用 | `RevolvingUtilizationOfUnsecuredLines` | 信用额度用得越满，流动性压力可能越高 |
| 历史逾期 | `NumberOfTimes90DaysLate` 等 | 历史逾期是非常强的风险信号 |
| 客户画像 | `age`, `NumberOfDependents` | 年龄、家庭负担可能影响风险 |

In [ ]:
summary = pd.DataFrame({
    "missing": df_raw.isna().sum(),
    "missing_rate": df_raw.isna().mean(),
    "dtype": df_raw.dtypes.astype(str),
})
display(summary)

## 4. 数据清洗

清洗思路：

1. 删除无意义索引列。
2. 对缺失值保留“是否缺失”的信息，再用中位数填补。
3. 对极端值做 1% 和 99% 分位截尾，避免少数异常值支配模型。

注意：这里不是为了把数据“修得好看”，而是为了让模型更稳定。

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "MonthlyIncome" in df.columns:
        df["monthly_income_missing"] = df["MonthlyIncome"].isna().astype(int)
        income_for_bins = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
        df["income_band"] = pd.cut(
            income_for_bins,
            bins=[-np.inf, 2500, 5000, 10000, 20000, np.inf],
            labels=[1, 2, 3, 4, 5],
        ).astype(float)

    if "NumberOfDependents" in df.columns:
        df["dependents_missing"] = df["NumberOfDependents"].isna().astype(int)

    if {"DebtRatio", "MonthlyIncome"}.issubset(df.columns):
        income_filled = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
        df["estimated_monthly_debt"] = df["DebtRatio"] * income_filled
        df["payment_burden_proxy"] = df["DebtRatio"]
        df["debt_to_income_ratio"] = df["DebtRatio"]

    late_cols = [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
    ]
    existing_late_cols = [col for col in late_cols if col in df.columns]
    if existing_late_cols:
        df["total_past_due_events"] = df[existing_late_cols].sum(axis=1)
        df["has_past_due"] = (df["total_past_due_events"] > 0).astype(int)

    if "RevolvingUtilizationOfUnsecuredLines" in df.columns:
        df["credit_utilization"] = df["RevolvingUtilizationOfUnsecuredLines"]
        df["high_credit_utilization"] = (
            df["RevolvingUtilizationOfUnsecuredLines"] > 0.8
        ).astype(int)

    if "age" in df.columns:
        df["age_band"] = pd.cut(
            df["age"],
            bins=[0, 30, 45, 60, 75, np.inf],
            labels=[1, 2, 3, 4, 5],
        ).astype(float)

    return df


def clip_outliers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in [c for c in df.columns if c != TARGET]:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = df[col].clip(lower, upper)
    return df


df = add_features(df_raw)
df = clip_outliers(df)

print("Cleaned shape:", df.shape)
display(df.head())

## 5. Risk EDA

EDA 的目的不是画图凑数量，而是验证风控直觉：

- 有逾期历史的人，未来风险是否更高？
- 信用额度使用率高的人，是否更危险？
- 收入、年龄、负债率是否和违约有关？

下面展示项目脚本已经生成的关键图表。

In [ ]:
figure_files = [
    "01_default_distribution.png",
    "02_age_vs_default.png",
    "03_RevolvingUtilizationOfUnsecuredLines_vs_default.png",
    "04_DebtRatio_vs_default.png",
    "05_MonthlyIncome_vs_default.png",
    "06_NumberOfTimes90DaysLate_vs_default.png",
    "07_correlation_heatmap.png",
]

for file_name in figure_files:
    path = FIGURE_DIR / file_name
    if path.exists():
        print(file_name)
        display(Image(filename=str(path)))

### EDA 小结

从风控角度看，比较重要的风险信号包括：

- 历史逾期次数：过去逾期越多，未来严重逾期风险越高。
- 信用额度使用率：额度使用越接近上限，说明资金压力可能越大。
- 负债率：负债越高，偿债压力越强。
- 收入：收入较低或收入缺失的客户，风险通常更高。

## 6. 建模准备

我们把数据分成训练集和测试集。

这里使用 `stratify=y`，是为了让训练集和测试集里的违约比例保持一致。

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

## 7. 训练并比较模型

这里比较三个模型：

1. **Logistic Regression**：风控里常见 baseline，可解释性强。
2. **Random Forest**：非线性树模型，能捕捉变量之间的复杂关系。
3. **Hist Gradient Boosting**：梯度提升树，通常在表格数据上表现较强。

评价指标：

- **AUC**：衡量模型区分好客户和坏客户的能力。
- **KS**：风控常用指标，衡量好坏客户分数分布的最大差异。

In [ ]:
def ks_score(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

numeric_features = X.columns.tolist()

scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        )
    ]
)

plain_preprocessor = ColumnTransformer(
    transformers=[("num", SimpleImputer(strategy="median"), numeric_features)]
)

models = {
    "logistic_regression": Pipeline([
        ("preprocess", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "random_forest": Pipeline([
        ("preprocess", plain_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=80,
            max_depth=6,
            min_samples_leaf=100,
            class_weight="balanced",
            n_jobs=2,
            random_state=RANDOM_STATE,
        )),
    ]),
    "hist_gradient_boosting": Pipeline([
        ("preprocess", plain_preprocessor),
        ("model", HistGradientBoostingClassifier(
            max_iter=120,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        )),
    ]),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_score = model.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "auc": roc_auc_score(y_test, y_score),
        "ks": ks_score(y_test, y_score),
    })

metrics = pd.DataFrame(results).sort_values("auc", ascending=False)
display(metrics)

### 模型结果解释

目前最好的模型是 `Hist Gradient Boosting`。

这说明违约风险和变量之间不是完全线性的，树模型能更好捕捉：

- 逾期记录和收入的组合风险
- 信用使用率和负债率的非线性关系
- 不同年龄段、收入段客户的风险差异

In [ ]:
best_name = metrics.iloc[0]["model"]
best_model = models[best_name]
y_score = best_model.predict_proba(X_test)[:, 1]

print("Best model:", best_name)
print("AUC:", roc_auc_score(y_test, y_score))
print("KS:", ks_score(y_test, y_score))

## 8. ROC 和 Precision-Recall

ROC 曲线展示模型区分好坏客户的能力。

Precision-Recall 曲线在样本不平衡问题里也很重要，因为违约客户占比只有约 6.7%。

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_score)
plt.title(f"ROC Curve - {best_name}")
plt.show()

precision, recall, _ = precision_recall_curve(y_test, y_score)
plt.figure(figsize=(6, 4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve - {best_name}")
plt.show()

## 9. 模型解释

模型解释要回答：

> 哪些变量让客户更可能被判断为高风险？

这里使用 permutation importance：如果打乱某个变量后 AUC 下降很多，说明模型很依赖这个变量。

In [ ]:
sample_size = min(5000, len(X_test))
X_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)
y_sample = y_test.loc[X_sample.index]

perm = permutation_importance(
    best_model,
    X_sample,
    y_sample,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=1,
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance.head(12))

In [ ]:
top = importance.head(12).sort_values("importance_mean")
plt.figure(figsize=(8, 5))
plt.barh(top["feature"], top["importance_mean"])
plt.xlabel("AUC decrease after permutation")
plt.title("Permutation Importance")
plt.tight_layout()
plt.show()

### 解释性结论

最重要的风险信号集中在：

- 历史逾期次数
- 信用额度使用率
- 年龄
- 开放信用账户数量
- 月收入
- 负债率

业务含义：

> 一个客户如果过去有多次逾期、信用额度使用率很高、收入弱且负债压力大，那么未来严重逾期风险更高。

## 10. 业务决策：审批阈值

模型输出的是风险分数。业务上需要设定规则：

```text
risk_score < threshold  -> approve
risk_score >= threshold -> reject or manual review
```

我们比较三个阈值：`0.3`、`0.5`、`0.7`。

In [ ]:
decision_rows = []
for threshold in [0.3, 0.5, 0.7]:
    approved = y_score < threshold
    rejected = ~approved
    decision_rows.append({
        "approval_threshold": threshold,
        "approval_rate": approved.mean(),
        "rejection_rate": rejected.mean(),
        "approved_default_rate": y_test[approved].mean(),
        "rejected_default_rate": y_test[rejected].mean(),
        "bad_capture_rate": y_test[rejected].sum() / y_test.sum(),
    })

business_decision = pd.DataFrame(decision_rows)
display(business_decision)

### 阈值怎么解释

- 阈值 `0.3`：更保守。拒绝更多客户，但能捕获更多坏客户。
- 阈值 `0.5`：相对折中。
- 阈值 `0.7`：更宽松。通过率高，但坏账风险也更高。

这个项目的重点不是简单追求最高准确率，而是把模型分数转化成业务可用的审批策略。

## 11. 高风险客群

从业务角度，高风险客户通常有这些特征：

- 有历史逾期记录
- 信用额度使用率过高
- 收入较低或收入缺失
- 负债压力较高
- 逾期次数多

这些变量可以作为贷前审批和贷后监控的 early warning signals。

In [ ]:
scored = X_test.copy()
scored["actual_default"] = y_test
scored["risk_score"] = y_score

segment_cols = ["has_past_due", "high_credit_utilization", "income_band", "age_band"]
for col in segment_cols:
    if col in scored.columns:
        print("\nSegment:", col)
        display(
            scored.groupby(col)
            .agg(
                customers=("actual_default", "size"),
                actual_default_rate=("actual_default", "mean"),
                avg_risk_score=("risk_score", "mean"),
            )
            .reset_index()
        )

## WOE Logistic Scorecard Benchmark

To make the project closer to a traditional credit risk workflow, I also built a compact WOE Logistic Scorecard benchmark.

This is not meant to replace the machine learning champion model. Its role is different:

- The scorecard is more interpretable and policy-friendly.
- The ML champion captures more complex non-linear risk patterns.
- Comparing both models shows the trade-off between transparency and predictive power.

In [ ]:
scorecard_features = pd.read_csv(REPORT_DIR / "scorecard_selected_features.csv")
scorecard_comparison = pd.read_csv(REPORT_DIR / "scorecard_vs_ml_comparison.csv")
scorecard_coefficients = pd.read_csv(REPORT_DIR / "scorecard_coefficients.csv")

display(scorecard_features)
display(scorecard_comparison)
display(scorecard_coefficients)

In [ ]:
path = FIGURE_DIR / "17_scorecard_coefficients.png"
if path.exists():
    display(Image(filename=str(path)))

### Scorecard interpretation

The WOE Logistic Scorecard achieved lower AUC/KS than the Hist Gradient Boosting model, but it is easier to explain. This is a realistic credit-risk trade-off:

- Use the ML model when stronger ranking performance is the priority.
- Use the scorecard benchmark when interpretability and policy transparency are more important.
- Use both together to understand whether the ML model is learning risk patterns that are consistent with traditional scorecard signals.

## 12. Decile / Lift / Gains Analysis

AUC tells us whether the model can separate good and bad borrowers overall. In credit risk, we also care about whether the model can **rank borrowers by risk**.

To test this, we sort borrowers by predicted default probability and split them into 10 equal-sized groups:

- Decile 1 = highest-risk 10% of borrowers
- Decile 10 = lowest-risk 10% of borrowers

Why this matters:

- If the top decile captures many future defaulters, the model is useful for manual review queues.
- Lift shows how much riskier a decile is compared with the portfolio average.
- Gains shows how many bad borrowers are captured as we review more of the population.

In [ ]:
decile_lift = pd.read_csv(REPORT_DIR / "decile_lift_gains.csv")
display(decile_lift)

In [ ]:
for file_name in ["12_default_rate_by_decile.png", "13_cumulative_gains.png"]:
    path = FIGURE_DIR / file_name
    if path.exists():
        print(file_name)
        display(Image(filename=str(path)))

### Decile interpretation

In this project, the top 10% highest-risk borrowers capture more than half of all observed defaulters. This is a strong signal that the model is useful for prioritizing high-risk customers, even if the final business decision still depends on risk appetite and approval targets.

## 13. WOE / IV Analysis

WOE and IV are common in credit scorecard development.

- **WOE (Weight of Evidence)** measures how each bin of a variable separates good and bad borrowers.
- **IV (Information Value)** summarizes how predictive a variable is overall.

Typical IV interpretation:

| IV range | Meaning |
|---|---|
| < 0.02 | Not useful |
| 0.02 - 0.10 | Weak |
| 0.10 - 0.30 | Medium |
| 0.30 - 0.50 | Strong |
| > 0.50 | Very strong, but should be checked for redundancy or leakage |

In [ ]:
iv_summary = pd.read_csv(REPORT_DIR / "iv_summary.csv")
woe_bins = pd.read_csv(REPORT_DIR / "woe_bins.csv")

display(iv_summary.head(15))
display(woe_bins.head(20))

In [ ]:
path = FIGURE_DIR / "14_information_value.png"
if path.exists():
    display(Image(filename=str(path)))

### WOE / IV interpretation

The strongest variables are historical delinquency and credit utilization. This aligns with credit-risk intuition: borrowers who have already missed payments, or who are using most of their credit limit, are much more likely to default in the future.

Some IV values are very high because engineered features are related to the same delinquency signals. In a production setting, we would check correlation and remove redundant variables before final scorecard modeling.

## 14. Calibration Analysis

A model can rank borrowers well but still produce probabilities that are too high or too low. Calibration checks whether predicted default probabilities match observed default rates.

Why this matters:

- If the model says a group has 10% PD, the observed default rate should be close to 10%.
- Calibration is important if risk scores are used in expected loss, pricing, or capital decisions.

In [ ]:
calibration_table = pd.read_csv(REPORT_DIR / "calibration_table.csv")
display(calibration_table)
print("Brier score:", calibration_table["brier_score"].iloc[0])

path = FIGURE_DIR / "15_calibration_plot.png"
if path.exists():
    display(Image(filename=str(path)))

## 15. SHAP Explainability

SHAP explains how each feature contributes to model predictions.

Permutation importance answers: **Which variables does the model rely on globally?**

SHAP answers: **How do variables push predictions higher or lower?**

This is useful for model governance and for explaining risk drivers to non-technical stakeholders.

In [ ]:
shap_status_path = REPORT_DIR / "shap_status.txt"
if shap_status_path.exists():
    print(shap_status_path.read_text())

shap_path = REPORT_DIR / "shap_importance.csv"
if shap_path.exists():
    shap_importance = pd.read_csv(shap_path)
    display(shap_importance.head(15))

fig_path = FIGURE_DIR / "16_shap_summary.png"
if fig_path.exists():
    display(Image(filename=str(fig_path)))

### SHAP interpretation

The SHAP results confirm the same core story as the EDA, IV, and permutation importance:

- Past-due behavior is the strongest risk driver.
- High revolving credit utilization increases predicted risk.
- Age, income, debt burden, and number of open credit lines also contribute to model predictions.

This makes the model story more credible because multiple independent analysis methods point to similar risk drivers.

## 16. 最终结论

这个项目完成了一个端到端信贷风控分析流程：

1. 用 Kaggle Give Me Some Credit 数据预测未来两年严重逾期。
2. 做了缺失值、异常值和风险特征工程。
3. 比较 Logistic Regression、Random Forest 和 Gradient Boosting。
4. 最佳模型 AUC 约 `0.868`，KS 约 `0.583`。
5. 用重要性分析解释风险因素。
6. 用审批阈值分析说明通过率和坏账风险的 trade-off。

简历表述：

> Built an end-to-end credit default risk model using the Kaggle Give Me Some Credit dataset with 150K borrowers. Engineered interpretable credit risk features, compared Logistic Regression, Random Forest, and Gradient Boosting models, achieving AUC 0.868 and KS 0.583. Translated model scores into approval-threshold decisions to balance approval rate and default risk.

## 17. 后续可以升级什么

如果继续增强项目，可以做：

- 加入 XGBoost 或 LightGBM。
- 安装 SHAP，做全局和单客户解释。
- 做 WOE / IV 分箱，更接近银行传统评分卡。
- 输出一个 credit score 分数，而不是只输出违约概率。
- 做更细的业务分群，例如年轻客户、高负债客户、历史逾期客户。